# Connect 4 AI - Training Demo

This notebook demonstrates how to load the environment, train the PPO agent, and evaluate its performance.

In [5]:
import sys
import os
import gymnasium as gym
import numpy as np
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env

# Add project root to path
sys.path.append(os.path.abspath('..'))

from envs.connect4_env import Connect4Env
from agent.train import SinglePlayerWrapper # We import the wrapper we defined in train.py (or we redefine it here)

ImportError: cannot import name 'SinglePlayerWrapper' from 'agent.train' (c:\Users\Simone\Documents\Code Projects\Pefforza_4\agent\train.py)

### 1. Setup Environment
We use the same wrapper logic as `train.py` to handle the opponent's moves internally.

In [ ]:
# Re-defining wrapper here for clarity if train.py class isn't easily importable (since it was inside a function)
class SinglePlayerWrapper(gym.Wrapper):
    def __init__(self, env):
        super().__init__(env)
        
    def step(self, action):
        # 1. Player (Agent) Move
        obs, reward, terminated, truncated, info = self.env.step(action)
        
        if terminated or truncated:
            return obs, reward, terminated, truncated, info
        
        # 2. Opponent Move (Random)
        valid_moves = [c for c in range(self.env.cols) if self.env.board[0, c] == 0]
        if not valid_moves:
            return obs, 0, True, False, {}
            
        import random
        opp_action = random.choice(valid_moves)
        obs, reward, terminated, truncated, info = self.env.step(opp_action)
        
        if terminated and reward == 1.0:
             reward = -1.0 # Agent lost
        
        return obs, reward, terminated, truncated, info

env = Connect4Env()
env = SinglePlayerWrapper(env)

### 2. Initialize Model
We use PPO with an MLP policy since the board is small (6x7).

In [ ]:
model = PPO("MlpPolicy", env, verbose=1)
print("Model initialized.")

### 3. Train
Train for a short duration (e.g. 1000 steps) to verify it runs.

In [ ]:
model.learn(total_timesteps=1000)
print("Training complete.")

### 4. Save and Test
Save the model and simulate one game.

In [ ]:
model.save("../agent/models/notebook_model")

obs, _ = env.reset()
done = False
print("Starting Game...")
env.render()

while not done:
    action, _ = model.predict(obs)
    obs, reward, done, truncated, info = env.step(action)
    env.render()
    
print(f"Game Over. Reward: {reward}")